In [1]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 1 - Install Required Packages
# ============================================

!pip install flask redis pika joblib pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 11.0 MB/s eta 0:00:00


In [2]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 2 - Import Required Libraries
# ============================================

from flask import Flask, request, jsonify
import redis
import pika
import joblib
import pandas as pd
import json
import requests
import threading

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 3 - Upload Pretrained Random Forest Model
# ============================================

from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Saving wellness_randomforest_pipeline.pkl to wellness_randomforest_pipeline.pkl
Uploaded files:
- wellness_randomforest_pipeline.pkl


In [5]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 4 - Load Pretrained Random Forest Model
# ============================================

MODEL_PATH = "wellness_randomforest_pipeline.pkl"

model_package = joblib.load(MODEL_PATH)

model = model_package["model"]
FEATURE_COLUMNS = model_package["features"]
RISK_MAPPING = model_package["risk_mapping"]

print("Pretrained Random Forest model loaded successfully!")
print("Number of ML features:", len(FEATURE_COLUMNS))
print("ML Features:")

for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i}. {feature}")

Pretrained Random Forest model loaded successfully!
Number of ML features: 14
ML Features:
1. facility_usage
2. dining_activity
3. event_participation
4. club_participation
5. residence_engagement
6. recreation_activity
7. social_interactions
8. communication_activity
9. sleep_quality
10. academic_engagement
11. campus_engagement_score
12. social_isolation_score
13. engagement_change_pct
14. rolling_3_week_engagement


In [6]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 5 - Redis Configuration
# ============================================

REDIS_HOST = "129.153.75.221"
REDIS_PORT = 6379
REDIS_USERNAME = "default"

print("Redis configuration loaded!")
print("Host:", REDIS_HOST)
print("Port:", REDIS_PORT)
print("Username:", REDIS_USERNAME)

Redis configuration loaded!
Host: 129.153.75.221
Port: 6379
Username: default


In [7]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 6 - Redis Authentication & Connection
# ============================================

from getpass import getpass

REDIS_PASSWORD = getpass("Enter Redis password: ")

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    username=REDIS_USERNAME,
    password=REDIS_PASSWORD,
    decode_responses=True,
    socket_connect_timeout=10,
    socket_timeout=10
)

print("Redis client created successfully!")

Enter Redis password: ··········
Redis client created successfully!


In [8]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 7 - Test Redis Connection
# ============================================

try:
    result = redis_client.ping()

    if result:
        print("Connected to Redis successfully!")
        print("Redis PING:", result)

except Exception as e:
    print("Redis connection failed!")
    print("Error:", e)

Connected to Redis successfully!
Redis PING: True


In [9]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 8 - RabbitMQ Configuration
# ============================================

RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"
RABBITMQ_VHOST = "/"

QUEUE_NAME = "student_wellness_predictions"

print("RabbitMQ configuration loaded!")
print("Host:", RABBITMQ_HOST)
print("Port:", RABBITMQ_PORT)
print("Username:", RABBITMQ_USERNAME)
print("Queue:", QUEUE_NAME)

RabbitMQ configuration loaded!
Host: 129.153.75.221
Port: 5672
Username: bytesmart_interns
Queue: student_wellness_predictions


In [10]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 9 - RabbitMQ Authentication & Connection
# ============================================

RABBITMQ_PASSWORD = getpass("Enter RabbitMQ password: ")

credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

rabbitmq_parameters = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=credentials,
    heartbeat=60,
    blocked_connection_timeout=300
)

rabbitmq_connection = pika.BlockingConnection(
    rabbitmq_parameters
)

rabbitmq_channel = rabbitmq_connection.channel()

rabbitmq_channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print("Connected to RabbitMQ successfully!")
print("Queue ready:", QUEUE_NAME)

Enter RabbitMQ password: ··········
Connected to RabbitMQ successfully!
Queue ready: student_wellness_predictions


In [11]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 10 - Verify RabbitMQ Connection
# ============================================

print("RabbitMQ connection open:", rabbitmq_connection.is_open)
print("RabbitMQ channel open:", rabbitmq_channel.is_open)

RabbitMQ connection open: True
RabbitMQ channel open: True


In [12]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 11 - Create Flask Application
# ============================================

app = Flask(__name__)

print("Flask application created successfully!")

Flask application created successfully!


In [13]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 12 - Home Route
# ============================================

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Predictive Campus Life Wellness Sentinel - Phase 5",
        "status": "API is running",
        "endpoint": "/predict"
    })

In [19]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 13 - Reliable RabbitMQ Publisher
# ============================================

def publish_prediction_to_rabbitmq(prediction_data):

    connection = None

    try:
        # Create a fresh RabbitMQ connection
        credentials = pika.PlainCredentials(
            RABBITMQ_USERNAME,
            RABBITMQ_PASSWORD
        )

        parameters = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            heartbeat=60,
            blocked_connection_timeout=300
        )

        connection = pika.BlockingConnection(parameters)

        channel = connection.channel()

        # Make sure the queue exists
        channel.queue_declare(
            queue=QUEUE_NAME,
            durable=True
        )

        # Enable publisher confirmations
        channel.confirm_delivery()

        message = json.dumps(prediction_data)

        channel.basic_publish(
            exchange="",
            routing_key=QUEUE_NAME,
            body=message,
            properties=pika.BasicProperties(
                delivery_mode=2,
                content_type="application/json"
            )
        )

        print("Prediction event published to RabbitMQ")

        return True

    except Exception as e:

        print("RabbitMQ publish failed:", str(e))

        return False

    finally:

        if connection is not None and connection.is_open:
            connection.close()

In [20]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# RabbitMQ Publisher Test
# ============================================

test_message = {
    "student_id": 9999,
    "predicted_risk": "Medium",
    "probabilities": {
        "Low": 10.0,
        "Medium": 70.0,
        "High": 20.0
    }
}

publish_status = publish_prediction_to_rabbitmq(test_message)

print("Publish Status:", publish_status)

Prediction event published to RabbitMQ
Publish Status: True


In [15]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 14 - Integrated /predict Endpoint
# ============================================

import hashlib


@app.route("/predict", methods=["POST"])
def predict():

    try:
        # --------------------------------------------
        # 1. Receive request through Flask
        # --------------------------------------------

        data = request.get_json()

        if not data:
            return jsonify({
                "error": "No JSON input provided"
            }), 400

        # --------------------------------------------
        # 2. Validate required ML features
        # --------------------------------------------

        missing_features = [
            feature
            for feature in FEATURE_COLUMNS
            if feature not in data
        ]

        if missing_features:
            return jsonify({
                "error": "Missing required features",
                "missing_features": missing_features
            }), 400

        # --------------------------------------------
        # 3. Create unique Redis cache key
        # --------------------------------------------

        cache_input = {
            feature: data[feature]
            for feature in FEATURE_COLUMNS
        }

        cache_string = json.dumps(
            cache_input,
            sort_keys=True
        )

        cache_hash = hashlib.md5(
            cache_string.encode()
        ).hexdigest()

        student_id = int(data.get("student_id", 0))

        cache_key = (
            f"student_wellness:"
            f"{student_id}:"
            f"{cache_hash}"
        )

        print("\n========================================")
        print("New prediction request")
        print("Student ID:", student_id)
        print("Cache Key:", cache_key)

        # --------------------------------------------
        # 4. Check Redis for cached result
        # --------------------------------------------

        cached_result = redis_client.get(cache_key)

        if cached_result:

            print("Redis Cache: HIT")
            print("Returning cached result")

            cached_response = json.loads(cached_result)

            cached_response["cache"] = {
                "hit": True,
                "key": cache_key
            }

            print("========================================\n")

            return jsonify(cached_response), 200

        # --------------------------------------------
        # Cache MISS
        # --------------------------------------------

        print("Redis Cache: MISS")
        print("Invoking Random Forest model...")

        # --------------------------------------------
        # 5. Invoke ML model
        # --------------------------------------------

        input_df = pd.DataFrame([data])

        # Keep only the 14 ML features
        input_df = input_df[FEATURE_COLUMNS]

        predicted_class = model.predict(input_df)[0]

        probabilities = model.predict_proba(input_df)[0]

        predicted_risk = RISK_MAPPING[predicted_class]

        probability_response = {
            "Low": round(float(probabilities[0]) * 100, 2),
            "Medium": round(float(probabilities[1]) * 100, 2),
            "High": round(float(probabilities[2]) * 100, 2)
        }

        print("Prediction:", predicted_risk)

        # --------------------------------------------
        # 6. Prepare result
        # --------------------------------------------

        prediction_result = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response
        }

        # --------------------------------------------
        # 7. Store result in Redis
        # --------------------------------------------

        redis_client.set(
            cache_key,
            json.dumps(prediction_result),
            ex=300
        )

        print("Prediction stored in Redis")
        print("Cache TTL: 300 seconds")

        # --------------------------------------------
        # 8. Publish event to RabbitMQ
        # --------------------------------------------

        rabbitmq_message = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response
        }

        publish_prediction_to_rabbitmq(
            rabbitmq_message
        )

        # --------------------------------------------
        # 9. Return result through Flask
        # --------------------------------------------

        response = {
            "student_id": student_id,
            "predicted_risk": predicted_risk,
            "probabilities": probability_response,
            "cache": {
                "hit": False,
                "stored": True,
                "ttl_seconds": 300,
                "key": cache_key
            },
            "rabbitmq": {
                "published": True,
                "queue": QUEUE_NAME
            }
        }

        print("Flask response generated")
        print("========================================\n")

        return jsonify(response), 200

    except Exception as e:

        print("Prediction error:", str(e))

        return jsonify({
            "error": str(e)
        }), 500

In [16]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 15 - Verify Flask Routes
# ============================================

print(app.url_map)

Map([<Rule '/static/<filename>' (HEAD, GET, OPTIONS) -> static>,
 <Rule '/' (HEAD, GET, OPTIONS) -> home>,
 <Rule '/predict' (POST, OPTIONS) -> predict>])


In [17]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 16 - Start Flask Server
# ============================================

def run_flask():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


flask_thread = threading.Thread(
    target=run_flask
)

flask_thread.daemon = True
flask_thread.start()

print("Flask server started successfully!")
print("Server running on port 5000")

Flask server started successfully!
Server running on port 5000


In [18]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 17 - First API Test (Cache MISS)
# ============================================

test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))


New prediction request
Student ID: 1001
Cache Key: student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1
Redis Cache: MISS
Invoking Random Forest model...


ERROR:pika.adapters.utils.io_services_utils:_AsyncBaseTransport._produce() failed, aborting connection: error=ConnectionResetError(104, 'Connection reset by peer'); sock=<socket.socket fd=53, family=2, type=1, proto=6, laddr=('172.28.0.12', 35070)>; Caller's stack:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pika/adapters/utils/io_services_utils.py", line 1177, in _on_socket_writable
    self._produce()
    ~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/pika/adapters/utils/io_services_utils.py", line 875, in _produce
    num_bytes_sent = self._sigint_safe_send(self._sock,
                                            self._tx_buffers[0])
  File "/usr/local/lib/python3.13/dist-packages/pika/adapters/utils/io_services_utils.py", line 84, in retry_sigint_wrap
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/pika/adapters/utils/io_services_utils.py", line 917, in _sigint_safe_send
    return sock.send(dat

Prediction: Medium
Prediction stored in Redis
Cache TTL: 300 seconds
Prediction error: Stream connection lost: ConnectionResetError(104, 'Connection reset by peer')
Status Code: 500
Response:
{
    "error": "Stream connection lost: ConnectionResetError(104, 'Connection reset by peer')"
}


In [21]:
# ============================================
# PHASE 5 - CLEAR PREVIOUS TEST CACHE
# ============================================

failed_cache_key = "student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1"

deleted = redis_client.delete(failed_cache_key)

print("Previous cache deleted:", deleted)

Previous cache deleted: 1


In [22]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 22 - Complete Flow Test (Cache MISS)
# ============================================

test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("Status Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))


New prediction request
Student ID: 1001
Cache Key: student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1
Redis Cache: MISS
Invoking Random Forest model...
Prediction: Medium
Prediction stored in Redis
Cache TTL: 300 seconds


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:38:22] "POST /predict HTTP/1.1" 200 -


Prediction event published to RabbitMQ
Flask response generated

Status Code: 200
Response:
{
    "cache": {
        "hit": false,
        "key": "student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1",
        "stored": true,
        "ttl_seconds": 300
    },
    "predicted_risk": "Medium",
    "probabilities": {
        "High": 29.25,
        "Low": 0.23,
        "Medium": 70.53
    },
    "rabbitmq": {
        "published": true,
        "queue": "student_wellness_predictions"
    },
    "student_id": 1001
}


In [23]:
# ============================================
# PHASE 5 - END-TO-END INTEGRATION
# Cell 39 - Redis Cache HIT Test
# ============================================

import requests
import json

# Same input used in the previous request
test_data = {
    "student_id": 1001,

    "facility_usage": 45,
    "dining_activity": 50,
    "event_participation": 30,
    "club_participation": 25,
    "residence_engagement": 40,
    "recreation_activity": 35,
    "social_interactions": 30,
    "communication_activity": 40,
    "sleep_quality": 60,
    "academic_engagement": 55,
    "campus_engagement_score": 42,
    "social_isolation_score": 58,
    "engagement_change_pct": -12,
    "rolling_3_week_engagement": 48
}

print("Sending the same prediction request again...")

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=test_data
)

print("\nStatus Code:", response.status_code)
print("Response:")
print(json.dumps(response.json(), indent=4))

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:55:11] "POST /predict HTTP/1.1" 200 -


Sending the same prediction request again...

New prediction request
Student ID: 1001
Cache Key: student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1
Redis Cache: HIT
Returning cached result


Status Code: 200
Response:
{
    "cache": {
        "hit": true,
        "key": "student_wellness:1001:7f681b5a01c6b6af974e91f3dd2651c1"
    },
    "predicted_risk": "Medium",
    "probabilities": {
        "High": 29.25,
        "Low": 0.23,
        "Medium": 70.53
    },
    "student_id": 1001
}
